<div dir="rtl">

# 01 - Extract: سحب أسعار الموبايلات من Jumia Egypt

ده أول Notebook في مشروع **Egypt Smartphone Price Tracker**، وهو مرحلة الـ **Extract** في الـ ETL Pipeline بتاعي.

**الهدف من الـ Notebook ده:**
- ألف على كل صفحات قسم Smartphones في Jumia.
- أسحب بيانات كل موبايل (الاسم، الماركة، السعر، السعر القديم، الخصم، التقييم، ...).
- أحفظ الداتا **زي ما هي من غير تنظيف** في فولدر `data/raw`.

**ليه مش بنضف الداتا هنا؟**
في الـ ETL كل مرحلة ليها شغلانة واحدة. لو حصل غلط في التنظيف بعدين، هيبقى عندي الداتا الخام أرجعلها من غير ما أسحب من الموقع تاني. التنظيف هيكون في Notebook `04_transform`.

</div>

<div dir="rtl">

## قبل ما أبدأ: قواعد الموقع (robots.txt)

فتحت `https://www.jumia.com.eg/robots.txt` ولقيت إن Jumia **بتسمح بالـ scraping** بشروط:

1. الـ **User-Agent** لازم يقول بوضوح إنه bot، ويكون فيه رابط أو وسيلة تواصل مع صاحبه.
2. ميتبعتش أكتر من **200 request في الدقيقة**.
3. الـ bot اللي بيتنكر في شكل متصفح (fake user-agent) بيتعمله block.

وكمان فيه مسارات ممنوعة زي `/mobapi/` (الـ API بتاعهم) وأي رابط فيه `sort=` أو `price=`.

**التزمت بالقواعد دي كالتالي:**
- استخدمت صفحات القسم العادية بس: `/smartphones/?page=1`، `?page=2`، ...، ودي مسموحة.
- حطيت في الـ User-Agent اسم المشروع ورابط الـ GitHub بتاعي.
- بستنى **ثانيتين** بين كل صفحة والتانية، يعني أقل من 30 request في الدقيقة، أقل بكتير من الحد المسموح.

</div>

<div dir="rtl">

## الخطوة 1: استدعاء المكتبات

| المكتبة | استخدمتها في إيه |
|---|---|
| `requests` | بتبعت طلب للموقع وترجع كود الصفحة (HTML) |
| `BeautifulSoup` | بتقرا الـ HTML وتسهل البحث جواه |
| `pandas` | بحط فيها الداتا في الآخر في جدول (DataFrame) وأحفظه |
| `time` | عشان أستنى بين كل request والتاني |
| `json` | بتحول نص JSON لـ dictionary في Python |
| `datetime` | عشان أسجل تاريخ ووقت السحب |
| `os` | عشان أعمل الفولدرات لو مش موجودة |

</div>

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import json
import os
from datetime import datetime

<div dir="rtl">

## الخطوة 2: الإعدادات

حطيت كل المتغيرات اللي ممكن أحتاج أغيرها بعدين في مكان واحد، بدل ما تبقى متفرقة في الكود.

- `base_url`: رابط قسم الموبايلات.
- `headers`: البيانات اللي بتتبعت مع كل request. أهم حاجة فيها الـ `User-Agent`، وده زي "الكارنيه" اللي بعرف بيه نفسي للموقع.
- `delay_seconds`: عدد الثواني اللي بستناها بين كل صفحة والتانية.
- `max_attempts`: عدد المحاولات لكل صفحة لو الموقع اتأخر في الرد.
- `scraped_at`: وقت السحب. بحطه مع كل منتج، عشان لما أسحب كل يوم أعرف كل سعر كان إمتى. ودي أساس فكرة **تتبع الأسعار**.

</div>

In [2]:
site_url = "https://www.jumia.com.eg"
base_url = site_url + "/smartphones/"

headers = {
    "User-Agent": "EgyptSmartphonePriceTracker/1.0 (+https://github.com/nepal-aly/egypt-smartphone-price-tracker)",
    "Accept-Language": "en-US,en;q=0.9",
}

delay_seconds = 2
max_attempts = 3

scraped_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Base URL:", base_url)
print("Scraped at:", scraped_at)

Base URL: https://www.jumia.com.eg/smartphones/
Scraped at: 2026-10-24 07:16:54


<div dir="rtl">

## الخطوة 3: تجربة أول صفحة

قبل ما أعمل loop على كل الصفحات، بجرب صفحة واحدة الأول وأتأكد إن كل حاجة شغالة. **دي عادة مهمة جداً في الـ scraping.**

- `requests.get(...)` بيبعت الطلب للموقع، والرد بيتحفظ في متغير اسمه `response`. النص بتاع الصفحة (الـ HTML) بيبقى في `response.text`.
- `params={"page": 1}` بيضيف `?page=1` على الرابط لوحده.
- `timeout=60` معناه لو الموقع مردش خلال 60 ثانية الطلب يقف، بدل ما الكود يفضل معلق.
- `response.status_code` هو رد الموقع:
  - **200** يعني تمام.
  - **403** يعني الموقع رفض الطلب (block).
  - **404** يعني الصفحة مش موجودة.

</div>

In [3]:
response = requests.get(base_url, params={"page": 1}, headers=headers, timeout=60)

print("Status code:", response.status_code)
print("Final URL:", response.url)

Status code: 200
Final URL: https://www.jumia.com.eg/smartphones/?page=1


<div dir="rtl">

## الخطوة 4: الداتا موجودة جاهزة كـ JSON جوه الصفحة

في أول تجربة دورت على كروت المنتجات في الـ HTML بنفس الـ classes اللي بشوفها في Chrome، وطلعلي **منتج واحد بس** بدل 40.

لما دورت ورا السبب، لقيت إن Jumia بتبعت **نسخة الموبايل** من الصفحة لأي حد مش متصفح (زي `requests`)، وشكل الـ HTML فيها مختلف عن نسخة الكمبيوتر.

ولما فتحت الـ HTML اللي راجع ودورت فيه، لقيت حاجة أحسن بكتير. في آخر الصفحة فيه `<script>` شكله كده:

```html
<script>
window.__INITIAL_STATE__={"viewData": {"products": [ ... ], "pagination": {"total": 47}}, ...}
</script>
```

ده **JSON** فيه كل بيانات المنتجات جاهزة ومترتبة، والموقع نفسه بيستخدمه عشان يرسم الصفحة. وقررت أعتمد عليه بدل الـ HTML لـ 3 أسباب:

1. **ثابت:** مش بيتأثر لو الموقع غير التصميم.
2. **أدق:** فيه معلومات مش ظاهرة في الكارت، زي الماركة والقسم وكود البائع.
3. **واحد على الموبايل والكمبيوتر:** نفس الشكل في النسختين.

الطريقة دي اسمها **Embedded JSON Scraping**. وهي مختلفة عن الـ API المخفي `/mobapi/` اللي Jumia منعته، لأني هنا بقرا نفس الصفحة العادية المسموحة.

</div>

<div dir="rtl">

## الخطوة 5: استخراج الـ JSON من الصفحة

**الخطوات:**
1. أجيب كل عناصر `<script>` في الصفحة بـ `soup.find_all("script")`.
2. ألف عليهم بـ loop، وأشوف أنهي واحد فيه `window.__INITIAL_STATE__=`.
3. آخد النص اللي **بعد** علامة `=` بـ `.split("window.__INITIAL_STATE__=")[1]`:
   - `split` بتقسم النص لـ list من حتتين عند الكلمة دي.
   - `[1]` معناها الحتة التانية، اللي بعد الكلمة.
4. أحول النص لـ dictionary في Python.

**ليه مش `json.loads` العادية؟**
لأن بعد الـ JSON في نفس الـ script فيه كود JavaScript تاني (`window.__CONFS__=...`)، و `json.loads` بتضرب Error لو لقت أي حاجة زيادة بعد الـ JSON.

عشان كده استخدمت `json.JSONDecoder().raw_decode(text)`. دي بتقرا **أول JSON كامل** وتقف عنده، وبترجع حاجتين:
- الـ dictionary.
- رقم المكان اللي وقفت عنده. ده مش محتاجاه، فحطيته في متغير اسمه `_`، ودي طريقة متعارف عليها في Python معناها "القيمة دي مش مستخدمة".

</div>

In [4]:
soup = BeautifulSoup(response.text, "html.parser")

state_text = None

for script in soup.find_all("script"):
    script_text = script.get_text()
    if "window.__INITIAL_STATE__=" in script_text:
        state_text = script_text.split("window.__INITIAL_STATE__=")[1]
        break

state, _ = json.JSONDecoder().raw_decode(state_text.strip())

print("Keys:", list(state.keys())[:10])

Keys: ['csrfToken', 'messages', 'response', 'viewData', 'head', 'view', 'activeLanguage', 'languages', 'languagesData', 'countryCode']


<div dir="rtl">

## الخطوة 6: الوصول للمنتجات وعدد الصفحات

الـ JSON عبارة عن dictionary جوه dictionary، واللي يهمني موجود جوه `viewData`:

| المسار | فيه إيه |
|---|---|
| `state["viewData"]["products"]` | **list** فيها المنتجات (40 منتج في الصفحة) |
| `state["viewData"]["pagination"]["total"]` | **عدد الصفحات الكلي** |
| `state["viewData"]["productsFound"]` | نص زي `1848 products found` |

كده مش محتاجة أدور على أزرار الصفحات في الـ HTML، لأن عدد الصفحات جاهز.

</div>

In [5]:
view_data = state["viewData"]

products = view_data["products"]
total_pages = view_data["pagination"]["total"]

print(view_data["productsFound"])
print("Products in page 1:", len(products))
print("Total pages:", total_pages)

1848 products found
Products in page 1: 40
Total pages: 47


<div dir="rtl">

## الخطوة 7: فهم المنتج الواحد

كل منتج عبارة عن dictionary. ده مكان كل معلومة فيه:

| العمود | مكانه في الـ JSON | مثال |
|---|---|---|
| `sku` | `product["sku"]` | `SA024MP34RNY7NAFAMZ` |
| `product_name` | `product["displayName"]` | `Samsung Galaxy A17 ...` |
| `brand` | `product["brand"]` | `Samsung` |
| `category` | آخر عنصر في `product["categories"]` | `Android Phones` |
| `price` | `product["prices"]["price"]` | `EGP 17,699.00` |
| `old_price` | `product["prices"]["oldPrice"]` | `EGP 21,638.00` |
| `discount` | `product["prices"]["discount"]` | `18%` |
| `rating` | `product["rating"]["average"]` | `4.5` |
| `reviews_count` | `product["rating"]["totalRatings"]` | `2` |
| `is_official_store` | `product["badges"]["main"]["name"]` | `Official Store` |
| `is_express` | `product["isShopExpress"]` | `True` |
| `url` | `product["url"]` | `/samsung-galaxy-....html` |
| `image_url` | `product["image"]` | رابط الصورة |

**ليه استخدمت `.get()`:**
مش كل المنتجات فيها كل المعلومات. مثلاً المنتج اللي مفيهوش خصم مش بيبقى فيه `oldPrice`. لو كتبت `product["prices"]["oldPrice"]` والمفتاح مش موجود، الكود هيضرب **KeyError** ويقف.

`.get()` بتحل المشكلة دي:
```python
product.get("brand")          # لو مش موجود بترجع None
product.get("rating", {})     # لو مش موجود بترجع dictionary فاضي {}
```
الـ `{}` الفاضي مهم لما يبقى فيه dictionary جوه dictionary، لأنه بيخليني أكمل وأكتب `.get("average")` عليه من غير Error.

</div>

In [6]:
product = products[0]

print("SKU:", product.get("sku"))
print("Name:", product.get("displayName"))
print("Brand:", product.get("brand"))
print("Categories:", product.get("categories"))
print("Prices:", product.get("prices"))
print("Rating:", product.get("rating"))
print("URL:", site_url + product.get("url"))

SKU: HO474MP26BRG1NAFAMZ
Name: Honor X5c Plus Dual SIM 4G 128GB/4GB - Midnight Black
Brand: Honor
Categories: ['Phones & Tablets', 'Mobile Phones', 'Smartphones', 'Android Phones']
Prices: {'rawPrice': '8599.00', 'price': 'EGP 8,599.00', 'priceEuro': '138.82', 'taxEuro': '17.05', 'oldPrice': 'EGP 9,460.00', 'oldPriceEuro': '152.71', 'discount': '9%', 'discountEuro': '13.90'}
Rating: {'average': 5, 'totalRatings': 12}
URL: https://www.jumia.com.eg/honor-x5c-plus-dual-sim-4g-128gb4gb-midnight-black-133885513.html


<div dir="rtl">

## الخطوة 8: تجهيز الـ Lists الفاضية

عملت **list لكل عمود**. ومع كل منتج، بضيف قيمة في كل list بـ `append`. في الآخر كل الـ lists بيبقى طولهم واحد، وبحولهم لـ DataFrame.

**ليه لازم أضيف قيمة (حتى لو `None`) في كل list مع كل منتج؟** عشان كل الـ lists تفضل بنفس الطول. لو منتج مالوش خصم ومضفتش حاجة في `old_price_list`، الصفوف هتتلخبط وسعر منتج هيروح لمنتج تاني.

</div>

In [7]:
site_list = []
sku_list = []
product_name_list = []
brand_list = []
category_list = []
price_list = []
old_price_list = []
discount_list = []
rating_list = []
reviews_count_list = []
is_official_store_list = []
is_express_list = []
is_sponsored_list = []
seller_id_list = []
url_list = []
image_url_list = []
page_list = []
scraped_at_list = []

failed_pages = []

<div dir="rtl">

## الخطوة 9: الـ Loop الأساسي - سحب كل الصفحات

دي أهم خطوة. فيها **loop خارجي** بيلف على الصفحات، وجواه حاجتين:

**1) loop المحاولات (`attempt`):**
في أول تشغيل قابلتني مشكلتين:
- صفحة 2 فشلت بـ `Read timed out`، لأن الموقع اتأخر في الرد.
- صفحة 11 رجعت بـ status 200، بس من غير منتجات، والكود وقف بـ `KeyError: 'products'`. ولما جربت نفس الصفحة تاني بعدها بشوية، رجعت عادي بـ 40 منتج. يعني المشكلة مؤقتة من ناحية الموقع.

عشان كده، كل صفحة بحاول فيها لحد `max_attempts` مرات، ومبعتبرش إن الصفحة نجحت غير لما **ألاقي فيها منتجات فعلاً**:
- لو حصل Error في الاتصال، أو الـ status مش 200، أو الصفحة مفيهاش منتجات → أستنى 5 ثواني وأحاول تاني (`continue`).
- لو لقيت منتجات → `break` وأكمل.
- لو كل المحاولات فشلت → أسجل رقم الصفحة في `failed_pages` وأكمل على الصفحة اللي بعدها، بدل ما الكود كله يقف.

`try / except` هي اللي بتمسك الـ Error لو النت فصل أو الموقع مردش.

استخدمت `state.get("viewData", {}).get("products")` بدل `state["viewData"]["products"]`، عشان لو المفتاح مش موجود ترجع `None` بدل ما الكود يقف. و `if products:` بتبقى `False` لو `products` بـ `None` أو list فاضية.

**2) loop المنتجات:**
بلف على كل منتج وأطلع منه المعلومات وأضيفها للـ lists.

وفي آخر كل صفحة بعمل `time.sleep(delay_seconds)` قبل الصفحة اللي بعدها، عشان مضغطش على الموقع.

**الوقت:** الموقع بياخد من 15 لـ 20 ثانية في الرد على كل صفحة، فالـ 47 صفحة بياخدوا حوالي **15 لـ 20 دقيقة**.

</div>

In [8]:
for page in range(1, total_pages + 1):

    # 1) Request the page and read the JSON (try up to max_attempts times)
    products = None

    for attempt in range(1, max_attempts + 1):

        try:
            response = requests.get(base_url, params={"page": page}, headers=headers, timeout=60)
        except requests.exceptions.RequestException as error:
            print("Page", page, "attempt", attempt, "-> error:", error)
            time.sleep(5)
            continue

        if response.status_code != 200:
            print("Page", page, "attempt", attempt, "-> status", response.status_code)
            time.sleep(5)
            continue

        soup = BeautifulSoup(response.text, "html.parser")

        state_text = None
        for script in soup.find_all("script"):
            script_text = script.get_text()
            if "window.__INITIAL_STATE__=" in script_text:
                state_text = script_text.split("window.__INITIAL_STATE__=")[1]
                break

        if state_text is not None:
            state, _ = json.JSONDecoder().raw_decode(state_text.strip())
            products = state.get("viewData", {}).get("products")

        if products:
            break

        print("Page", page, "attempt", attempt, "-> no products found in the page")
        products = None
        time.sleep(5)

    if products is None:
        failed_pages.append(page)
        continue

    # 3) Loop over the products
    for product in products:

        prices = product.get("prices", {})
        rating = product.get("rating", {})
        badge = product.get("badges", {}).get("main", {})
        categories = product.get("categories", [])

        if len(categories) > 0:
            category = categories[-1]
        else:
            category = None

        if badge.get("name") == "Official Store":
            is_official_store = True
        else:
            is_official_store = False

        site_list.append("jumia")
        sku_list.append(product.get("sku"))
        product_name_list.append(product.get("displayName"))
        brand_list.append(product.get("brand"))
        category_list.append(category)
        price_list.append(prices.get("price"))
        old_price_list.append(prices.get("oldPrice"))
        discount_list.append(prices.get("discount"))
        rating_list.append(rating.get("average"))
        reviews_count_list.append(rating.get("totalRatings"))
        is_official_store_list.append(is_official_store)
        is_express_list.append(product.get("isShopExpress", False))
        is_sponsored_list.append(product.get("isSponsored", False))
        seller_id_list.append(product.get("sellerId"))
        url_list.append(site_url + product.get("url", ""))
        image_url_list.append(product.get("image"))
        page_list.append(page)
        scraped_at_list.append(scraped_at)

    print("Page", page, "of", total_pages, "->", len(products), "products | total so far:", len(sku_list))

    time.sleep(delay_seconds)

print()
print("Done. Total products:", len(sku_list))
print("Failed pages:", failed_pages)

Page 1 of 47 -> 40 products | total so far: 40
Page 2 of 47 -> 40 products | total so far: 80
Page 3 of 47 -> 40 products | total so far: 120
Page 4 of 47 -> 40 products | total so far: 160
Page 5 of 47 -> 40 products | total so far: 200
Page 6 of 47 -> 40 products | total so far: 240
Page 7 of 47 -> 40 products | total so far: 280
Page 8 of 47 -> 40 products | total so far: 320
Page 9 of 47 -> 40 products | total so far: 360
Page 10 of 47 -> 40 products | total so far: 400
Page 11 of 47 -> 40 products | total so far: 440
Page 12 of 47 -> 40 products | total so far: 480
Page 13 of 47 -> 40 products | total so far: 520
Page 14 of 47 -> 40 products | total so far: 560
Page 15 of 47 -> 40 products | total so far: 600
Page 16 of 47 -> 40 products | total so far: 640
Page 17 of 47 -> 40 products | total so far: 680
Page 18 of 47 -> 40 products | total so far: 720
Page 19 of 47 -> 40 products | total so far: 760
Page 20 of 47 -> 40 products | total so far: 800
Page 21 of 47 -> 40 products | 

<div dir="rtl">

## الخطوة 10: تجميع الداتا في DataFrame

عملت **dictionary**: كل key هو اسم العمود، وكل value هي الـ list بتاعته. وبعدين `pd.DataFrame(...)` بيحوله لجدول.

</div>

In [9]:
df = pd.DataFrame({
    "site": site_list,
    "sku": sku_list,
    "product_name": product_name_list,
    "brand": brand_list,
    "category": category_list,
    "price": price_list,
    "old_price": old_price_list,
    "discount": discount_list,
    "rating": rating_list,
    "reviews_count": reviews_count_list,
    "is_official_store": is_official_store_list,
    "is_express": is_express_list,
    "is_sponsored": is_sponsored_list,
    "seller_id": seller_id_list,
    "url": url_list,
    "image_url": image_url_list,
    "page": page_list,
    "scraped_at": scraped_at_list,
})

df.head()

,site,sku,product_name,brand,category,price,old_price,discount,rating,reviews_count,is_official_store,is_express,is_sponsored,seller_id,url,image_url,page,scraped_at
0,jumia,HO474MP26BRG1NAFAMZ,Honor X5c Plus Dual SIM 4G 128GB/4GB - Midnigh...,Honor,Android Phones,"EGP 8,599.00","EGP 9,460.00",9%,5.0,12,False,True,False,NaN,https://www.jumia.com.eg/honor-x5c-plus-dual-s...,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,1,2026-10-24 07:16:54
1,jumia,HO474MP2CA3HTNAFAMZ,Honor X5c Plus Dual SIM 4G 128GB/4GB - Ocean Cyan,Honor,Android Phones,"EGP 8,599.00","EGP 9,460.00",9%,4.4,20,False,True,False,NaN,https://www.jumia.com.eg/honor-x5c-plus-dual-s...,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,1,2026-10-24 07:16:54
2,jumia,HO474MP27U60XNAFAMZ,"Honor T10 AI Translator Phone 64GB 4GB , 4G Du...",Honor,Android Phones,"EGP 6,975.00","EGP 7,699.00",9%,1.0,1,False,True,False,201886.0,https://www.jumia.com.eg/honor-t10-ai-translat...,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,1,2026-10-24 07:16:54
3,jumia,OP872MP1WHCI3NAFAMZ,OPPO A6 -6.7-Inch 128GB/4GB Dual SIM 4G Mobile...,OPPO,Android Phones,"EGP 12,525.00","EGP 13,780.00",9%,4.4,7,False,True,False,NaN,https://www.jumia.com.eg/oppo-a6-6.7-inch-128g...,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,1,2026-10-24 07:16:54
4,jumia,SA024MP220DBFNAFAMZ,"Samsung Samsung Galaxy A16 Dual SIM, 128GB, 4G...",Samsung,Android Phones,"EGP 11,599.00","EGP 12,717.00",9%,4.0,1,False,True,False,202698.0,https://www.jumia.com.eg/samsung-samsung-galax...,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,1,2026-10-24 07:16:54


<div dir="rtl">

## الخطوة 11: فحص سريع للداتا

قبل الحفظ، ببص بصة سريعة أتأكد إن الداتا منطقية:

- `df.shape`: عدد الصفوف والأعمدة.
- `df.isnull().sum()`: عدد الخانات الفاضية في كل عمود. طبيعي `old_price` و `discount` و `rating` يبقى فيهم فاضي، لأن مش كل المنتجات عليها خصم أو تقييم. **بس `product_name` و `price` المفروض ميبقاش فيهم فاضي.**
- `df["sku"].duplicated().sum()`: عدد المنتجات المكررة. ممكن يطلع تكرار بسيط لو الموقع رتب المنتجات من جديد وقت السحب، وده هعالجه في مرحلة التنظيف.
- `df["brand"].value_counts()`: أكتر الماركات عدداً.

</div>

In [10]:
print("Shape:", df.shape)
print()
print("Missing values:")
print(df.isnull().sum())
print()
print("Duplicated SKUs:", df["sku"].duplicated().sum())

Shape: (1848, 18)

Missing values:
site                   0
sku                    0
product_name           0
brand                  0
category               0
price                  0
old_price            236
discount             236
rating                 0
reviews_count          0
is_official_store      0
is_express             0
is_sponsored           0
seller_id            202
url                    0
image_url              0
page                   0
scraped_at             0
dtype: int64

Duplicated SKUs: 0


In [11]:
df["brand"].value_counts().head(10)

brand
Samsung    459
Honor      237
XIAOMI     222
Infinix    197
OPPO       177
realme     157
Apple      109
vivo        95
Renewed     53
Tecno       49
Name: count, dtype: int64

<div dir="rtl">

## الخطوة 12: حفظ الداتا الخام

بحفظ نسختين في `data/raw`:

- **CSV**: ده اللي هشتغل عليه في مرحلة التنظيف، لأنه خفيف وأي أداة تقدر تقراه.
- **Excel**: عشان يتفتح بسهولة ويتعرض كنموذج.

**اسم الملف فيه التاريخ** (مثلاً `jumia_raw_2026-10-24.csv`). كده لما أشغل الـ Notebook كل يوم، كل يوم بيبقى ليه ملف لوحده والقديم مش بيتمسح، وده اللي هيخليني أتابع تغير الأسعار.

- `os.makedirs(..., exist_ok=True)` بيعمل الفولدر لو مش موجود، ولو موجود مبيعملش حاجة.
- `index=False` عشان عمود الأرقام (0، 1، 2، ...) بتاع pandas ميتحفظش.
- `encoding="utf-8-sig"` عشان لو فيه أي حروف عربي تظهر صح لما الـ CSV يتفتح في Excel.

</div>

In [12]:
os.makedirs("data/raw", exist_ok=True)

today = datetime.now().strftime("%Y-%m-%d")

csv_path = "data/raw/jumia_raw_" + today + ".csv"
excel_path = "data/raw/jumia_raw_" + today + ".xlsx"

df.to_csv(csv_path, index=False, encoding="utf-8-sig")
df.to_excel(excel_path, index=False)

print("Saved:", csv_path)
print("Saved:", excel_path)

Saved: data/raw/jumia_raw_2026-10-24.csv
Saved: data/raw/jumia_raw_2026-10-24.xlsx


<div dir="rtl">

## الخلاصة

في الـ Notebook ده:

1. قريت قواعد الموقع والتزمت بيها (User-Agent واضح، وانتظار بين الصفحات).
2. جربت صفحة واحدة الأول قبل الـ loop.
3. لقيت إن الداتا موجودة كـ JSON جاهز جوه الصفحة (`window.__INITIAL_STATE__`)، واستخدمته بدل الـ HTML.
4. جبت عدد الصفحات من الـ JSON، ولفيت على كل الصفحات وسحبت 18 عمود لكل موبايل.
5. اتعاملت مع المنتجات اللي ناقصها معلومات، وضفت إعادة محاولة للصفحات اللي الموقع بيتأخر فيها.
6. حفظت الداتا الخام بتاريخ اليوم.

**الخطوة الجاية:** `02_extract_noon.ipynb`

</div>